# Notebook 3 · Images, a pretrained model, and an app

Part of the hands-on session **Deep Learning, Hands-on**. The page with the activities: **https://gradlabs.in/learn/deep-learning/**

1. **A CNN** on CIFAR-10: 60,000 small colour photos of 10 kinds of things.
2. **Fine-tune** a pretrained network on the same photos, and compare.
3. **Zero-shot** with CLIP: no training at all.
4. **An app** you can share with a link.
5. **The challenge:** beat your CNN.

⚡ This notebook needs the GPU: **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom.

In [ ]:
import torch
print("PyTorch", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print("✓ GPU found:", torch.cuda.get_device_name(0))
else:
    print("No GPU. It still works, just slower.")
    print("To turn it on: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

## 1 · The photos

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
plain = transforms.ToTensor()
train_set = datasets.CIFAR10("data", train=True, download=True, transform=plain)
test_set = datasets.CIFAR10("data", train=False, download=True, transform=plain)
train_data = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_data = DataLoader(test_set, batch_size=256, num_workers=2)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.flat, range(16)):
    photo, label = train_set[i]
    ax.imshow(photo.permute(1, 2, 0)); ax.set_title(CLASSES[label]); ax.axis("off")
plt.show()
print("Each photo is 32 × 32 pixels, with 3 numbers per pixel: red, green and blue.")

## 2 · A CNN from scratch

- `nn.Conv2d(3, 32, 3, padding=1)`: 32 filters of 3 × 3, sliding over the 3 colour layers, like the filter on the page.
- `nn.MaxPool2d(2)`: shrinks the picture by half, keeping the biggest number in each 2 × 2 square. Later filters then see a bigger area.

In [ ]:
def make_cnn(width=32, extra_layer=False):
    layers = [
        nn.Conv2d(3, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),               # 32 → 16
        nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 16 → 8
        nn.Conv2d(2 * width, 4 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8 → 4
    ]
    if extra_layer:
        layers += [nn.Conv2d(4 * width, 4 * width, 3, padding=1), nn.ReLU()]
    layers += [nn.Flatten(), nn.Linear(4 * width * 4 * 4, 256), nn.ReLU(), nn.Linear(256, 10)]
    return nn.Sequential(*layers).to(device)

def train_one_round(model, data, optimizer, loss_fn):
    model.train()
    for images, labels in data:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

def accuracy(model, data):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for images, labels in data:
            images, labels = images.to(device), labels.to(device)
            right += (model(images).argmax(1) == labels).sum().item(); total += len(labels)
    return 100 * right / total

def train(model, train_data, rounds, lr=0.001, test=None):
    test = test or test_data
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for r in range(1, rounds + 1):
        train_one_round(model, train_data, optimizer, loss_fn)
        print(f"round {r}: test {accuracy(model, test):.1f}%")
    return model

In [ ]:
cnn = train(make_cnn(), train_data, rounds=5)   # about 1 minute on a T4 GPU
cnn_score = accuracy(cnn, test_data)
print(f"Your CNN: {cnn_score:.1f}% right on the test set")

Around 70%. Not bad from zero, in a minute. Now let's borrow a network that has already seen over a million photos.

## 3 · Fine-tune a pretrained network

**ResNet-18** was trained on ImageNet: 1.2 million photos of 1,000 kinds of things. We keep all its layers (🔒 frozen) and swap the last one for a new layer with 10 outputs.

Two changes to the photos, because that's what ResNet learned from: bigger (128 × 128) and with ImageNet's colour averages taken away (`Normalize`). To keep it quick, we fine-tune on 10,000 of the 50,000 training photos.

In [ ]:
from torchvision import models

resnet_look = transforms.Compose([
    transforms.Resize(128), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_big = datasets.CIFAR10("data", train=True, transform=resnet_look)
test_big = datasets.CIFAR10("data", train=False, transform=resnet_look)
train_big_data = DataLoader(Subset(train_big, range(10_000)), batch_size=128, shuffle=True, num_workers=2)
test_data_big = DataLoader(test_big, batch_size=256, num_workers=2)

pretrained = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in pretrained.parameters():
    p.requires_grad = False                   # 🔒 keep what it learned
pretrained.fc = nn.Linear(512, 10)            # ✏️ a new last layer for our 10 kinds
pretrained = pretrained.to(device)
print("Weights that will learn:", sum(p.numel() for p in pretrained.parameters() if p.requires_grad))

In [ ]:
pretrained = train(pretrained, train_big_data, rounds=3, test=test_data_big)   # about 3 minutes on a T4 GPU
fine_score = accuracy(pretrained, test_data_big)
print(f"From scratch: {cnn_score:.1f}%   Fine-tuned: {fine_score:.1f}%")

It learned only 5,130 numbers, from a fifth of the photos, and still beat the CNN. That's transfer learning.

## 4 · Zero-shot with CLIP

**CLIP** learned from hundreds of millions of photos with captions. We give it each test photo and ten sentences, "a photo of a plane", "a photo of a car" …, and it picks the sentence that fits best. **No training at all.**

The first run downloads the model (about 600 MB), which takes a minute.

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
sentences = [f"a photo of a {c}" for c in CLASSES]

raw_test = datasets.CIFAR10("data", train=False)          # the photos as plain pictures
def clip_guess(pictures):
    inputs = processor(text=sentences, images=pictures, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        return clip(**inputs).logits_per_image.softmax(1).cpu()   # a chance for each sentence

pictures = [raw_test[i][0] for i in range(500)]
truth = torch.tensor([raw_test[i][1] for i in range(500)])
chances = torch.cat([clip_guess(pictures[i:i + 100]) for i in range(0, 500, 100)])
print(f"CLIP, with no training: {100 * (chances.argmax(1) == truth).float().mean():.1f}% right on 500 test photos")

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(13, 3))
for ax, i in zip(axes, range(6)):
    top = chances[i].argmax().item()
    ax.imshow(pictures[i]); ax.axis("off")
    ax.set_title(f"{CLASSES[top]} {100 * chances[i][top]:.0f}%", color="black" if top == truth[i] else "red")
plt.show()

✏️ **Try:** change `sentences` to other kinds, like `["a photo of an animal", "a photo of a vehicle"]`, and run the two cells again. You just made a new classifier by typing.

## 5 · Share it as an app

Gradio turns a function into a web page. `share=True` prints a public link (it lasts about 3 days). Upload any photo, from your phone too.

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

def model_scores(image):
    x = resnet_look(image.convert("RGB")).unsqueeze(0).to(device)
    pretrained.eval()
    with torch.no_grad():
        return pretrained(x).softmax(1)[0].tolist()

def predict(image):
    scores = model_scores(image)
    return {name: score for name, score in zip(CLASSES, scores)}

gr.Interface(predict, gr.Image(type="pil"), gr.Label(num_top_classes=3),
             title="What's in this photo?").launch(share=True)

## 6 · 🏆 The challenge: the leaderboard

Teams of 3 or 4, **15 minutes**. Change **one thing at a time**, run the cell, and send your score to the leaderboard on screen. Send as often as you like: your best score counts.

**Two prizes:** the best score, and the **biggest jump** from your first score to your best.

| Idea | What to change |
|---|---|
| Wider | `width=64` |
| Deeper | `extra_layer=True` |
| Longer | `rounds=10` (watch the test score each round) |
| Flip | `flip=True`: flipped copies of photos while training (**data augmentation**) |
| Rate | `lr=0.0005` or `lr=0.003` |

First, your team name and the score to beat:

In [ ]:
TEAM = "Your team name"     # ✏️ the same name every time you send
print(f"The starter CNN: {cnn_score:.1f}%")

In [ ]:
# ✏️ Change these
width, extra_layer, rounds, flip, lr = 32, False, 5, False, 0.001

look = transforms.Compose(([transforms.RandomHorizontalFlip()] if flip else []) + [transforms.ToTensor()])
challenge_data = DataLoader(datasets.CIFAR10("data", train=True, transform=look), batch_size=128, shuffle=True, num_workers=2)
mine = train(make_cnn(width, extra_layer), challenge_data, rounds, lr)
print(f"Your score: {accuracy(mine, test_data):.1f}%   (to beat: {cnn_score:.1f}%)")

### 📤 Send your score

The code is on the leaderboard on screen. Every team is scored on the same 10,000 test photos, so it's fair. Run this cell after each try you want on the board.

In [ ]:
import requests

CODE = ""   # ✏️ the code on the screen

score = round(accuracy(mine, test_data), 1)
changed = f"width={width}, extra_layer={extra_layer}, rounds={rounds}, flip={flip}, lr={lr}"

if TEAM.strip() in ("", "Your team name"):
    print("✏️ Type your team name in the TEAM cell above, run it, then run this cell again.")
elif not CODE.strip():
    print("✏️ Type the code from the screen in CODE, then run this cell again.")
else:
    try:
        reply = requests.post("https://gradlabs.in/api/leaderboard/deep-learning",
                              json={"code": CODE, "team": TEAM, "score": score, "changed": changed}, timeout=20)
        answer = reply.json()
    except Exception:
        answer = {"error": "Couldn't reach the leaderboard. Check the internet and try again."}
    if answer.get("ok"):
        print(f"✓ On the board: {answer['team']}, {answer['score']}%")
    else:
        print("Not sent:", answer.get("error", "something went wrong"))

## ✅ Done

You trained a CNN, fine-tuned a pretrained network, used a foundation model with no training, and shared an app. Come back to the page any time: **https://gradlabs.in/learn/deep-learning/**